### Notebook for pulling random l0 observations or darks and random lines within to compare dark pedestal effects 

In [ ]:
import os
import glob
import random

import numpy as np
import pandas as pd
from astropy.io import fits

import matplotlib.pyplot as plt 

In [ ]:
folder_path = "/home/bekah/m3-pipeline-dev/data/dark/darks_global"

random_seed = None
if random_seed is not None:
    random.seed(random_seed)

obs_cal_mapping_path = "/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/obs_cal_info.csv"
temperature_table_path = "/home/bekah/m3-pipeline-dev/obs_to_cal_file_mapping/m3_detector_temperature.tab"
dark_dir = "/home/bekah/m3-pipeline-dev/data/dark/darks_global"

N_FILES = 500   #how many files to look at 
N_LINES = 40   # how many lines w/in the file

In [ ]:
fits_files = glob.glob(os.path.join(folder_path, "*.fits"))
print(f"Found {len(fits_files)} FITS files in: {folder_path}")

n_select = min(N_FILES, len(fits_files))
if n_select == 0:
    raise FileNotFoundError(f"No .fits files found in {folder_path}")
if len(fits_files) < N_FILES:
    print(f"Warning: only {len(fits_files)} files available, selecting all of them.")

selected_files = random.sample(fits_files, n_select)

print("\nrandom files:")
for f in selected_files:
    print(f" - {f}")

In [ ]:
metadata_df = pd.read_csv(obs_cal_mapping_path)
temp_df = pd.read_fwf(temperature_table_path, header=None, names=["obs_id", "temperature"])


def get_obs_id(filepath):
    fname = os.path.basename(filepath)
    return fname.split("_l0")[0]


def get_temperature(obs_id):
    match = temp_df.loc[temp_df["obs_id"] == obs_id.upper(), "temperature"]
    if match.empty:
        return None
    return match.iloc[0]

In [ ]:
## MODIFY BELOW TO LOOK AT L0 OBS DSS FILES, RN IT'S FOR DARK IMAGES

def dark_subtract(obs_id, obs_path):
    # meta = metadata_df[metadata_df["obs_id"] == obs_id.upper()]

    # if len(meta) == 0:
    #     raise ValueError(f"'{obs_id}' not found in obs/cal mapping table.")
    # elif len(meta) > 1:
    #     best_idx = meta["version"].str.extract(r"(\d+)")[0].astype(int).idxmax()
    #     meta = meta.loc[[best_idx]]

    #dark_id = meta["dark_signal_id"].iloc[0].lower()

    dark_path = os.path.join(dark_dir, f"{obs_id}_l0.fits")
    with fits.open(dark_path) as hdul:
        dark = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]
    #dark = dark.mean(axis=0)  # shape: (bands, cols)

    # with fits.open(obs_path) as hdul:
    #     obs_image = hdul[0].data.transpose(1, 0, 2)[5:-5, :, :]  # (lines, bands, cols)

    # obs_image = obs_image - dark[np.newaxis, :, :]

    # return obs_image
    return dark, obs_id

In [ ]:
def line_band_stats(line_data):
    combined_slices = np.concatenate([line_data[318:320], line_data[1:4]])

    return {
        "mean_cols_1_3": np.nanmean(line_data[1:3]),
        "val_col_1": line_data[1] if line_data.shape[0] > 1 else np.nan,
        "mean_cols_20_300": np.nanmean(line_data[20:300]),
        "median_cols_20_300": np.nanmedian(line_data[20:300]),
        "p90_cols_20_300": np.nanpercentile(line_data[20:300], 90),
        "mean_cols_318_320": np.nanmean(line_data[318:320]),
        "median_allcols": np.nanmedian(combined_slices), 
    }

In [ ]:
records = []

for f in selected_files:
    fname = os.path.basename(f)
    obs_id = get_obs_id(f)

    try:


        obs_image, obs_id = dark_subtract(obs_id, f)
        n_lines, n_bands, n_cols = obs_image.shape

        temp = get_temperature(obs_id)
        if temp is None:
            print(f"Warning: no temperature found for obs_id '{obs_id}' ({fname}); skipping.")
            continue
            
        n_lines_to_sample = min(N_LINES, n_lines)
        sampled_lines = random.sample(range(n_lines), n_lines_to_sample)

        for line_idx in sampled_lines:
            for band_idx in range(n_bands):
                line_data = obs_image[line_idx, band_idx, :]
                stats = line_band_stats(line_data)
                records.append({
                    "obs_id": obs_id,
                    "temperature": temp,
                    "band": band_idx,
                    "line": line_idx,
                    **stats,
                })

        print(f"Processed {fname}: {n_lines_to_sample} lines x {n_bands} bands")

        del obs_image

    except Exception as e:
        print(f"Skipping {fname}: {e}")

## Combine into a single results DataFrame

In [ ]:
stats_df = pd.DataFrame(records)
print(f"\nFinal DataFrame shape: {stats_df.shape}")
stats_df.head(20)

In [ ]:
output_path = "m3_line_band_stats_dark.csv"
stats_df.to_csv(output_path, index=False)
print(f"saved {output_path}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(stats_df['median_cols_20_300'],stats_df['mean_cols_318_320'], c=stats_df['band'],s=1)
plt.plot(x, y, color='blue', linestyle='-', label='x = y')
plt.xlim(500, 1200)
plt.ylim(500, 1500)
plt.colorbar()
plt.ylabel('column 1 value')
plt.xlabel('center col mean (20:300)')
plt.title('random lines from 40 DSS L0 images')
#plt.savefig('centervsdarkcol_1.png')


In [ ]:
plt.figure(figsize=(12, 10))

band_df = stats_df[(stats_df['band'] == 5) & (stats_df['temperature'] < 170)]

plt.scatter(band_df['median_cols_20_300'],band_df['mean_cols_318_320'], c=band_df['temperature'],s=1)
plt.scatter(band_df['median_cols_20_300'],band_df['mean_cols_1_3'], c=band_df['temperature'],s=1)
plt.scatter(band_df['median_cols_20_300'],band_df['val_col_1'], c=band_df['temperature'],s=1)

xmin = 500
xmax = 600
plt.xlim(xmin, xmax)
plt.ylim(xmin, 800)

x = np.linspace(xmin, xmax, 100)
y = x  
plt.plot(x, y, color='blue', linestyle='-', label='x = y')

plt.grid()

plt.colorbar()
plt.ylabel('various dark col values')
plt.xlabel('median_cols_20_300')
#plt.savefig('centervsdarkcol_1.png')


In [ ]:
plt.figure(figsize=(12, 10))

band_df = stats_df[(stats_df['band'] ==64) & (stats_df['temperature'] < 170)]

plt.scatter(band_df['temperature'],band_df['mean_cols_318_320'], c=band_df['median_cols_20_300'],s=1)
plt.scatter(band_df['temperature'],band_df['mean_cols_1_3'], c=band_df['median_cols_20_300'],s=1)
plt.scatter(band_df['temperature'],band_df['val_col_1'], c=band_df['median_cols_20_300'],s=1)

band_df = stats_df[(stats_df['band'] ==60) & (stats_df['temperature'] < 170)]

plt.scatter(band_df['temperature'],band_df['mean_cols_318_320'], c=band_df['median_cols_20_300'],s=1)
plt.scatter(band_df['temperature'],band_df['mean_cols_1_3'], c=band_df['median_cols_20_300'],s=1)
plt.scatter(band_df['temperature'],band_df['val_col_1'], c=band_df['median_cols_20_300'],s=1)


xmin = 144
xmax = 171
plt.xlim(xmin, xmax)
plt.ylim(500, 900)

x = np.linspace(xmin, xmax, 100)
y = x  
plt.plot(x, y, color='blue', linestyle='-', label='x = y')

plt.grid()

plt.colorbar()
plt.ylabel('various dark col values')
plt.xlabel('median_cols_20_300')
#plt.savefig('centervsdarkcol_1.png')


In [ ]:

plt.figure(figsize=(10, 6))

plt.xlim(500, 1500)
plt.ylim(500, 2200)

plt.scatter(stats_df['median_allcols'],stats_df['mean_cols_1_3'], c=stats_df['temperature'],s=1)

plt.colorbar()
x = np.linspace(500, 1500, 100)
y = x  

plt.plot(x, y, color='blue', linestyle='-', label='x = y')

plt.xlabel("median cols 20:300") 
plt.ylabel("mean cols 1:3") 
plt.title("ranodm lines from 500 dark signal obs") 

plt.savefig("darksig_alldarkcols_vs_mediancenter") 

In [ ]:
plt.figure(figsize=(10, 6))

plt.xlim(500, 600)
plt.ylim(500, 700)

cool_df = stats_df[stats_df['temperature'] < 155] 

warm_df = stats_df[stats_df['temperature'] >= 155] 

plt.scatter(warm_df['median_cols_20_300'],warm_df['mean_cols_1_3'], c=warm_df['band'],s=1, alpha=.1)

#plt.scatter(cool_df['median_cols_20_300'],cool_df['mean_cols_1_3'], c=cool_df['temperature'],s=1, alpha=.9)

plt.colorbar()

In [ ]:
plt.scatter(stats_df['band'], stats_df['mean_cols_1_3']/stats_df['median_cols_20_300'], s=1, c=stats_df['temperature']) 
plt.ylim(-1,4)

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(stats_df['mean_cols_20_300'],stats_df['mean_cols_318_320'], c=stats_df['temperature'],s=1, alpha=.4)

plt.ylabel('mean cols 318:320')
plt.xlabel('center col mean (20:300)')
plt.title('random lines from 40 DSS L0 images')

#plt.ylim(10,-60)
plt.colorbar()
#plt.savefig('rightdarkcols_vs_centervals_tempcolor.png')


In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(stats_df['band'], stats_df['mean_cols_1_3']/stats_df['mean_cols_20_300'],s=1, alpha=.4, c='blue', label='col 1')

#plt.scatter(stats_df['p90_cols_20_300'],stats_df['mean_cols_1_3'],s=1, alpha=.4, c='orange', label='col 1:3 mean')

#plt.ylim(10,-60)

plt.legend()
plt.xlabel("p90 center cols")
plt.ylabel("dark cols") 
#plt.savefig('darkcols_vs_centervals.png')
#plt.colorbar()

In [ ]:
plt.scatter(stats_df['band'], stats_df['val_col_1'],s=1, alpha=.4, c='blue')
plt.scatter(stats_df['band'], stats_df['mean_cols_1_3'],s=1, alpha=.4, c='orange')

In [ ]:
plt.scatter(stats_df['band'], stats_df['val_col_1']/stats_df['mean_cols_20_300'],s=1, alpha=.4, c='blue')
plt.scatter(stats_df['band'], stats_df['mean_cols_1_3']/stats_df['mean_cols_20_300'],s=1, alpha=.4, c='orange')


In [ ]:

grouped = stats_df.groupby('band')[['mean_cols_1_3', 'mean_cols_20_300']].mean()
band_ratios = grouped['mean_cols_1_3'] / grouped['mean_cols_20_300']

band_ratios.plot(kind='line', figsize=(10, 6), label='1:3')

grouped2 = stats_df.groupby('band')[['val_col_1', 'mean_cols_20_300']].mean()
band_ratios2 = grouped2['val_col_1'] / grouped2['mean_cols_20_300']

band_ratios2.plot(kind='line', figsize=(10, 6), label='1')


grouped3 = stats_df.groupby('band')[['mean_cols_318_320', 'mean_cols_20_300']].mean()
band_ratios3 = grouped3['mean_cols_318_320'] / grouped3['mean_cols_20_300']

band_ratios3.plot(kind='line', figsize=(10, 6), label='318:320')

plt.title('Mean Ratio of Dark Cols to Cols 20-300 per Band')
plt.xlabel('Band')
plt.ylabel('Ratio')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig("band_center_to_darkcols_ratios.png")
plt.show()


In [ ]:
filtered_df = stats_df[stats_df['temperature'] >= 156]

grouped = filtered_df.groupby('band')[['median_allcols', 'mean_cols_20_300']].mean()
band_ratios = grouped['median_allcols'] / grouped['mean_cols_20_300']

band_ratios.plot(kind='line', figsize=(10, 6), label='median_allcols, > 156 K')

# grouped2 = filtered_df.groupby('band')[['val_col_1', 'mean_cols_20_300']].mean()
# band_ratios2 = grouped2['val_col_1'] / grouped2['mean_cols_20_300']
# band_ratios2.plot(kind='line', figsize=(10, 6), label='1, > 160 K')

# grouped3 = filtered_df.groupby('band')[['mean_cols_318_320', 'mean_cols_20_300']].mean()
# band_ratios3 = grouped3['mean_cols_318_320'] / grouped3['mean_cols_20_300']
# band_ratios3.plot(kind='line', figsize=(10, 6), label='318:320, > 160 K')


filtered_df = stats_df[stats_df['temperature'] < 153]

grouped = filtered_df.groupby('band')[['median_allcols', 'mean_cols_20_300']].mean()
band_ratios = grouped['median_allcols'] / grouped['mean_cols_20_300']

band_ratios.plot(kind='line', figsize=(10, 6), label='median_allcols, < 153 K')

grouped2 = filtered_df.groupby('band')[['val_col_1', 'mean_cols_20_300']].mean()
band_ratios2 = grouped2['val_col_1'] / grouped2['mean_cols_20_300']
band_ratios2.plot(kind='line', figsize=(10, 6), label='1, < 153 K')

grouped3 = filtered_df.groupby('band')[['mean_cols_318_320', 'mean_cols_20_300']].mean()
band_ratios3 = grouped3['mean_cols_318_320'] / grouped3['mean_cols_20_300']
band_ratios3.plot(kind='line', figsize=(10, 6), label='318:320, < 153 K')

plt.title('Mean Ratio of Dark Cols to Mean Center per Band')
plt.xlabel('Band')
plt.ylabel('Ratio')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
#plt.savefig("band_center_to_darkcols_ratios.png")

plt.show()

In [ ]:
filtered_df = stats_df[stats_df['temperature'] >= 160]

grouped = filtered_df.groupby('band')[['median_allcols', 'median_cols_20_300']].mean()
band_ratios = grouped['median_allcols'] / grouped['median_cols_20_300']
band_ratios.plot(kind='line', figsize=(10, 6), label='median_allcols, > 156 K')

grouped2 = filtered_df.groupby('band')[['val_col_1', 'median_cols_20_300']].mean()
band_ratios2 = grouped2['val_col_1'] / grouped2['median_cols_20_300']
band_ratios2.plot(kind='line', figsize=(10, 6), label='1, > 156 K')

grouped3 = filtered_df.groupby('band')[['mean_cols_318_320', 'median_cols_20_300']].mean()
band_ratios3 = grouped3['mean_cols_318_320'] / grouped3['median_cols_20_300']
band_ratios3.plot(kind='line', figsize=(10, 6), label='318:320, > 156 K')


filtered_df = stats_df[stats_df['temperature'] < 150]

grouped = filtered_df.groupby('band')[['median_allcols', 'median_cols_20_300']].mean()
band_ratios = grouped['median_allcols'] / grouped['median_cols_20_300']

band_ratios.plot(kind='line', figsize=(10, 6), label='median_allcols, < 156 K')

grouped2 = filtered_df.groupby('band')[['val_col_1', 'median_cols_20_300']].mean()
band_ratios2 = grouped2['val_col_1'] / grouped2['median_cols_20_300']
band_ratios2.plot(kind='line', figsize=(10, 6), label='1, < 156 K')

grouped3 = filtered_df.groupby('band')[['mean_cols_318_320', 'median_cols_20_300']].mean()
band_ratios3 = grouped3['mean_cols_318_320'] / grouped3['median_cols_20_300']
band_ratios3.plot(kind='line', figsize=(10, 6), label='318:320, < 156 K')

plt.axvline(64)
    
plt.title('Mean Ratio of Dark Cols to Median Center per Band for Dark Obs')
plt.xlabel('Band')
plt.ylabel('Ratio')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
#plt.savefig("band_center_median_to_darkcols_ratios_darks.png")

plt.show()

In [ ]:
filtered_df = stats_df[stats_df['temperature'] < 160]

grouped = filtered_df.groupby('band')[['mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios = grouped['median_cols_20_300']

band_ratios.plot(kind='line', figsize=(10, 6), label='< 160 K')


filtered_df = stats_df[stats_df['temperature'] > 160]

grouped = filtered_df.groupby('band')[['mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios2 = grouped['median_cols_20_300']

band_ratios2.plot(kind='line', figsize=(10, 6), label='> 160 K')


((band_ratios2/band_ratios)).plot(kind='line', figsize=(10, 6), label='ratio')
plt.axvline(12, c='red')
plt.axvline(49, c='red')

plt.legend()
#plt.savefig('median_center_vals_fortempgroups.png')

In [ ]:
filtered_df = stats_df[stats_df['temperature'] < 160]

grouped = filtered_df.groupby('band')[['mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios = grouped['median_cols_20_300']

filtered_df = stats_df[stats_df['temperature'] > 160]

grouped = filtered_df.groupby('band')[['mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios2 = grouped['median_cols_20_300']

np.abs((band_ratios2/band_ratios)/2).plot(kind='line', figsize=(10, 6), label='< 160 K', color='red')

plt.axvline(12, c='red')
plt.axvline(49, c='red')

grouped2 = filtered_df.groupby('band')[['val_col_1', 'median_cols_20_300']].mean()
band_ratios2 = np.abs(grouped2['val_col_1'] / grouped2['median_cols_20_300'])
(band_ratios2*7).plot(kind='line', figsize=(10, 6), label='1, > 160 K')

In [ ]:
filtered_df = stats_df[stats_df['temperature'] < 160]
grouped = filtered_df.groupby('band')[['val_col_1','mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios = grouped['median_cols_20_300']-grouped['mean_cols_1_3']

filtered_df = stats_df[stats_df['temperature'] > 160]
grouped = filtered_df.groupby('band')[['val_col_1','mean_cols_1_3', 'median_cols_20_300']].mean()
band_ratios2 = grouped['median_cols_20_300']-grouped['mean_cols_1_3']

fig, ax1 = plt.subplots(figsize=(10, 6))

band_ratios.plot(kind='line', ax=ax1, label='< 160 K', color='C0')
band_ratios2.plot(kind='line', ax=ax1, label='> 160 K', color='C1')
ax1.set_ylabel('median_cols_20_300')

ax2 = ax1.twinx()
(band_ratios2 / band_ratios).plot(kind='line', ax=ax2, label='ratio (>160K / <160K)', color='C2')
ax2.set_ylabel('Ratio')

ax1.axvline(12, c='red')
ax1.axvline(49, c='red')
#ax2.axhline(1,c='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')

plt.tight_layout()
plt.savefig('median_center_vals_fortempgroups_ratio_darkpedcorr.png')

In [ ]:
plt.scatter(stats_df['temperature'], stats_df['mean_cols_1_3'], s=1) 

In [ ]:
plt.scatter(stats_df['temperature'], stats_df['val_col_1'], s=1) 

In [ ]:
plt.scatter(stats_df['mean_cols_318_320'], stats_df['median_cols_20_300'], s=1, c=stats_df['temperature']) 

In [ ]:
metadata_df = pd.read_csv(obs_cal_mapping_path)


In [ ]:
metadata_df.keys()

In [ ]:
metadata_df['dark_diff'] = metadata_df['obs_temperature'] - metadata_df['dark_signal_temp']

In [ ]:
plt.scatter(metadata_df['obs_temperature'], metadata_df['dark_diff'], s=1)
plt.axhline(0, c='red', ls=":")
plt.ylim(-1,1)
plt.xlabel("obs temperature")
plt.ylabel("kelvin diff from observation minus dark")
plt.savefig("dark_vs_obs_diff_temp.png")

In [ ]:
plt.hist(metadata_df['dark_diff'], bins=200) 
plt.xlim(xmax=2)
plt.xlabel("kelvin diff from observation minus dark") 
plt.savefig("dark_vs_obs_hist_temp.png")

In [ ]:
 metadata_df[metadata_df['obs_id'] == "m3g20090811t013030".upper()]['dark_diff']

In [ ]:
 metadata_df[metadata_df['obs_id'] == "m3g20090714t122932".upper()]['dark_diff']

In [ ]:
metadata_df[metadata_df['obs_id'] == "m3g20090617t045633".upper()]['dark_diff']

In [ ]:
metadata_df[metadata_df['obs_id'] == "m3g20090423t152245".upper()]['dark_diff']

In [ ]:
metadata_df[metadata_df['obs_id'] == "m3g20090214t074247".upper()]['dark_diff']

In [ ]:
metadata_df[metadata_df['obs_id'] == "m3g20090118t022705".upper()]['dark_diff']

M3G20090214T07424 146.79K -0.06
M3G20090811T013030 147.47K -0.12
M3G20090423T152245 150.82K -0.39
M3G20090714T122932 157.35K 0.0
M3G20090118T022705 158.78K -0.15
M3G20090617T045633 167.90K -0.09
